# 01 — Analyse request URL

Load captured LinkedIn URLs from `mitm_http_captures` and explore structure step by step.

Run Jupyter from the repo root so `from core.db import SessionLocal` works.

In [1]:
from collections import Counter
from urllib.parse import parse_qs, urlparse

from sqlalchemy import text

from core.db import SessionLocal

query = text("SELECT request_url FROM mitm_http_captures ORDER BY captured_at_ms DESC")

with SessionLocal() as session:
    rows = session.execute(query).fetchall()

request_urls = [row[0] for row in rows]
print(len(request_urls), "urls loaded")
print(request_urls[0])

2670 urls loaded
https://www.linkedin.com/flagship-web/rsc-action/actions/server-stream-request?sduiid=com.linkedin.sdui.realtimeDefaultHandler&payload=%7B%22requestId%22%3A%22com.linkedin.sdui.realtimeDefaultHandler%22%2C%22serverRequest%22%3A%7B%22requestId%22%3A%22com.linkedin.sdui.realtimeDefaultHandler%22%2C%22requestedArguments%22%3A%7B%22%24type%22%3A%22proto.sdui.actions.requests.RequestedArguments%22%2C%22payload%22%3A%7B%22sessionId%22%3A%7B%22key%22%3A%22realtimeSessionId%22%2C%22namespace%22%3A%22MemoryNamespace%22%7D%2C%22lastHeartbeatTimestamp%22%3A%7B%22key%22%3A%22realtimeHeartbeat%22%2C%22namespace%22%3A%22MemoryNamespace%22%7D%7D%2C%22requestedStateKeys%22%3A%5B%7B%22key%22%3A%7B%22value%22%3A%7B%22%24case%22%3A%22id%22%2C%22id%22%3A%22realtimeSessionId%22%7D%7D%7D%2C%7B%22key%22%3A%7B%22value%22%3A%7B%22%24case%22%3A%22id%22%2C%22id%22%3A%22realtimeHeartbeat%22%7D%7D%7D%5D%2C%22requestMetadata%22%3A%7B%22%24type%22%3A%22proto.sdui.common.RequestMetadata%22%7D%7D%2C%2

## hostname / netloc

In [2]:
for url in request_urls[:5]:
    parsed = urlparse(url)
    print(parsed.hostname, parsed.netloc)

print()

print(Counter(urlparse(url).hostname for url in request_urls))
print(Counter(urlparse(url).netloc for url in request_urls))

www.linkedin.com www.linkedin.com
www.linkedin.com www.linkedin.com
www.linkedin.com www.linkedin.com
www.linkedin.com www.linkedin.com
www.linkedin.com www.linkedin.com

Counter({'www.linkedin.com': 2670})
Counter({'www.linkedin.com': 2670})


## path

In [4]:
from pprint import pprint
pprint(Counter(urlparse(url).path for url in request_urls).most_common(20))

[('/voyager/api/graphql', 1695),
 ('/flagship-web/rsc-action/actions/component', 326),
 ('/flagship-web/rsc-action/actions/server-stream-request', 187),
 ('/voyager/api/voyagerMessagingGraphQL/graphql', 112),
 ('/flagship-web/rsc-action/actions/server-request', 88),
 ('/flagship-web/rsc-action/actions/app-config', 42),
 ('/flagship-web/feed/', 32),
 ('/voyager/api/messaging/dash/presenceStatuses', 29),
 ('/voyager/api/voyagerOrganizationDashPageMailbox/', 29),
 ('/voyager/api/voyagerMessagingDashConversationNudges', 28),
 ('/voyager/api/voyagerMessagingDashSecondaryInbox', 28),
 ('/voyager/api/voyagerIdentityDashNotificationCards', 28),
 ('/voyager/api/voyagerNotificationsDashBadgingItemCounts', 21),
 ('/voyager/api/voyagerGlobalAlerts', 21),
 ('/flagship-web/jobs/view/4417153139/', 1),
 ('/flagship-web/mynetwork', 1),
 ('/flagship-web/jobs/view/4429571858', 1),
 ('/flagship-web/jobs/search-results', 1)]


## filter to one path

Pick the path from the counts above that looks like job component requests.

In [5]:
component_path = "/flagship-web/rsc-action/actions/component"
component_urls = [url for url in request_urls if urlparse(url).path == component_path]

print(len(component_urls), "urls with that path")
print(component_urls[0])

326 urls with that path
https://www.linkedin.com/flagship-web/rsc-action/actions/component?componentId=com.linkedin.sdui.generated.jobseeker.dsl.impl.aboutTheCompanyForJobDetails&sduiid=com.linkedin.sdui.generated.jobseeker.dsl.impl.aboutTheCompanyForJobDetails&parentSpanId=5DAcd85p6PI%3D


## query params

In [6]:
query_key_sets = Counter(
    frozenset(parse_qs(urlparse(url).query).keys()) for url in component_urls
)
print(query_key_sets)

for url in component_urls[:3]:
    params = parse_qs(urlparse(url).query)
    print({k: len(v) for k, v in params.items()})
    print(params)
    print("-" * 40)

Counter({frozenset({'parentSpanId', 'sduiid', 'componentId'}): 326})
{'componentId': 1, 'sduiid': 1, 'parentSpanId': 1}
{'componentId': ['com.linkedin.sdui.generated.jobseeker.dsl.impl.aboutTheCompanyForJobDetails'], 'sduiid': ['com.linkedin.sdui.generated.jobseeker.dsl.impl.aboutTheCompanyForJobDetails'], 'parentSpanId': ['5DAcd85p6PI=']}
----------------------------------------
{'componentId': 1, 'sduiid': 1, 'parentSpanId': 1}
{'componentId': ['com.linkedin.sdui.generated.jobseeker.dsl.impl.similarJobs'], 'sduiid': ['com.linkedin.sdui.generated.jobseeker.dsl.impl.similarJobs'], 'parentSpanId': ['oxVWhUWE4/4=']}
----------------------------------------
{'componentId': 1, 'sduiid': 1, 'parentSpanId': 1}
{'componentId': ['com.linkedin.sdui.generated.jobseeker.dsl.impl.jobAlertToggle'], 'sduiid': ['com.linkedin.sdui.generated.jobseeker.dsl.impl.jobAlertToggle'], 'parentSpanId': ['gfzS9bcksd0=']}
----------------------------------------


## sduiid vs componentId

In [7]:
mismatches = []
for url in component_urls:
    params = parse_qs(urlparse(url).query)
    if params["componentId"][0] != params["sduiid"][0]:
        mismatches.append(url)

print(len(mismatches), "mismatches")
if mismatches:
    print(mismatches[0])

0 mismatches


## componentId

In [9]:
component_ids = [parse_qs(urlparse(url).query)["componentId"][0] for url in component_urls]

print(component_ids[0])
print()

# looks like they share a prefix — strip it and count suffixes
prefix = "com.linkedin.sdui.generated.jobseeker.dsl.impl."
suffixes = [cid[len(prefix):] if cid.startswith(prefix) else cid for cid in component_ids]
pprint(Counter(suffixes))

com.linkedin.sdui.generated.jobseeker.dsl.impl.aboutTheCompanyForJobDetails

Counter({'aboutTheCompanyForJobDetails': 33,
         'jobAlertToggle': 33,
         'peopleWhoCanHelp': 33,
         'premiumCompanyInsightsForJobDetails': 33,
         'aboutTheJob': 33,
         'resumeReview': 33,
         'premiumApplicantInsightsForJobDetails': 33,
         'similarJobs': 31,
         'manageJobBanner': 31,
         'howYouFitGuide': 30,
         'jobMatch': 3})


## parentSpanId

In [10]:
parent_span_ids = [parse_qs(urlparse(url).query)["parentSpanId"][0] for url in component_urls]

print(len(set(parent_span_ids)), "distinct values")
print(Counter(parent_span_ids).most_common(10))
print()
print(parent_span_ids[0])

326 distinct values
[('5DAcd85p6PI=', 1), ('oxVWhUWE4/4=', 1), ('gfzS9bcksd0=', 1), ('l2pzq3iMfzM=', 1), ('71yu5naYIOQ=', 1), ('r+1aoBedopA=', 1), ('qiRcCfnqLoU=', 1), ('Yh4SL7pBIZY=', 1), ('jhUXcDKQ2+8=', 1), ('WFiE8BH9024=', 1)]

5DAcd85p6PI=
